# 5. rdMMPA
## Add replacement pattern / divide fragment
- rdMMPA는 "분해"만 함, "교체"는 replacement_library와 결합해야 함
- 다음 단계: core의 [*:1] 자리에 replacement_library candidate를 붙여 새 분자 재조립하는 함수 필요 (Chem.molzip 등 조사)
- 원본 SMILES + core + chain을 함께 기록하는 구조 필요

In [ ]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 38.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 52 (delta 9), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 222.80 KiB | 8.91 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/laidd-2026
/content/laidd-2026


In [ ]:
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates

print("도구 로드 확인 완료")

도구 로드 확인 완료


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMMPA

test_smiles = "c1ccc(cc1)[N+](=O)[O-]"
mol = Chem.MolFromSmiles(test_smiles)

fragments = rdMMPA.FragmentMol(mol)
print("생성된 fragment 개수:", len(fragments))
for f in fragments:
    print(f)

생성된 fragment 개수: 1
(None, <rdkit.Chem.rdchem.Mol object at 0x7bf2f932bdf0>)


In [ ]:
for core, chain in fragments:
    core_smiles = Chem.MolToSmiles(core) if core is not None else None
    chain_smiles = Chem.MolToSmiles(chain) if chain is not None else None
    print(f"core: {core_smiles}, chain: {chain_smiles}")

core: None, chain: O=[N+]([O-])[*:1].c1ccc([*:1])cc1


In [ ]:
fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
print("생성된 fragment 개수:", len(fragments2))
for core, chain in fragments2:
    print(f"core: {core}, chain: {chain}")

생성된 fragment 개수: 1
core: , chain: O=[N+]([O-])[*:1].c1ccc([*:1])cc1


In [ ]:
# Tox21 데이터에서 좀 더 복잡한 분자 하나 가져오기
from src.tools.data_prep import load_tox21_clean
data = load_tox21_clean()

# 원자 수가 적당히 많은 분자 하나 선택 (너무 작지 않은 것)
sample_smiles = None
for s in data['smiles_train']:
    m = Chem.MolFromSmiles(s)
    if m and 15 <= m.GetNumAtoms() <= 30:
        sample_smiles = s
        break

print("선택된 분자:", sample_smiles)
mol2 = Chem.MolFromSmiles(sample_smiles)

fragments3 = rdMMPA.FragmentMol(mol2, maxCuts=1, resultsAsMols=False)
print("fragment 개수:", len(fragments3))
for core, chain in fragments3[:5]:  # 처음 5개만 확인
    print(f"core: {core}, chain: {chain}")

[06:02:43] WARNING: not removing hydrogen atom without neighbors
[06:02:43] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:02:44] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:02:44] WARNING: not removing hydrogen atom without neighbors


KeyError: 'smiles_train'

In [ ]:
import importlib
import src.tools.data_prep
importlib.reload(src.tools.data_prep)
from src.tools.data_prep import load_tox21_clean

data = load_tox21_clean()
print(data.keys())  # 'smiles_train'이 포함되어 있는지 먼저 확인

[06:03:59] WARNING: not removing hydrogen atom without neighbors
[06:03:59] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:03:59] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:03:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:03:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:04:00] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:04:00] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:04:00] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:04:00] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:04:00] WARNING: not removing hydrogen atom without neighbors


dict_keys(['X_train', 'y_train', 'w_train', 'X_valid', 'y_valid', 'w_valid', 'X_test', 'y_test', 'w_test', 'task_cols', 'invalid_smiles'])


In [ ]:
!cat src/tools/data_prep.py

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split

TOX21_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def _is_valid_smiles(smiles):
    return Chem.MolFromSmiles(smiles) is not None

def _smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol)
    return np.array(fp)

def load_tox21_clean(test_size=0.3, valid_ratio=0.5, random_state=42):
    df = pd.read_csv(TOX21_URL)
    df['valid'] = df['smiles'].apply(_is_valid_smiles)

    n_total, n_valid = len(df), df['valid'].sum()
    print(f"전체: {n_total}개, 파싱 성공: {n_valid}개, 파싱 실패(제외): {n_total - n_valid}개")

    invalid_smiles = df[~df['valid']]['smiles'].tolist()
    df_clean = df[df['valid']].reset_index(drop=True)

    task_cols = [c for c in df.columns if c not in [

In [ ]:
%%writefile src/tools/data_prep.py
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split

TOX21_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def _is_valid_smiles(smiles):
    return Chem.MolFromSmiles(smiles) is not None

def _smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol)
    return np.array(fp)

def load_tox21_clean(test_size=0.3, valid_ratio=0.5, random_state=42):
    df = pd.read_csv(TOX21_URL)
    df['valid'] = df['smiles'].apply(_is_valid_smiles)

    n_total, n_valid = len(df), df['valid'].sum()
    print(f"전체: {n_total}개, 파싱 성공: {n_valid}개, 파싱 실패(제외): {n_total - n_valid}개")

    invalid_smiles = df[~df['valid']]['smiles'].tolist()
    df_clean = df[df['valid']].reset_index(drop=True)

    task_cols = [c for c in df.columns if c not in ['smiles', 'mol_id', 'valid']]

    y = df_clean[task_cols].fillna(0).values.astype(np.float32)
    w = (~df_clean[task_cols].isna()).values.astype(np.float32)
    X = np.stack(df_clean['smiles'].apply(_smiles_to_ecfp).values)
    smiles_arr = df_clean['smiles'].values

    indices = np.arange(len(X))
    train_idx, temp_idx = train_test_split(indices, test_size=test_size, random_state=random_state)
    valid_idx, test_idx = train_test_split(temp_idx, test_size=valid_ratio, random_state=random_state)

    return {
        'X_train': X[train_idx], 'y_train': y[train_idx], 'w_train': w[train_idx],
        'smiles_train': smiles_arr[train_idx],
        'X_valid': X[valid_idx], 'y_valid': y[valid_idx], 'w_valid': w[valid_idx],
        'smiles_valid': smiles_arr[valid_idx],
        'X_test': X[test_idx], 'y_test': y[test_idx], 'w_test': w[test_idx],
        'smiles_test': smiles_arr[test_idx],
        'task_cols': task_cols,
        'invalid_smiles': invalid_smiles,
    }

Overwriting src/tools/data_prep.py


In [ ]:
import importlib
import src.tools.data_prep
importlib.reload(src.tools.data_prep)
from src.tools.data_prep import load_tox21_clean

data = load_tox21_clean()
print(data.keys())  # smiles_train 포함 확인

[06:06:58] WARNING: not removing hydrogen atom without neighbors
[06:06:59] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:06:59] Explicit valence for atom # 20 Al, 6, is greater than permitted
[06:07:00] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
dict_keys(['X_train', 'y_train', 'w_train', 'smiles_train', 'X_valid', 'y_valid', 'w_valid', 'smiles_valid', 'X_test', 'y_test', 'w_test', 'smiles_test', 'task_cols', 'invalid_smiles'])


In [ ]:
!git add src/tools/data_prep.py
!git commit -m "Fix: restore SMILES fields in data_prep.py (accidentally reverted)"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@82f3c8da80bc.(none)')
Everything up-to-date


In [ ]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"
!git commit -m "Fix: restore SMILES fields in data_prep.py (accidentally reverted)"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main a31b4e3] Fix: restore SMILES fields in data_prep.py (accidentally reverted)
 1 file changed, 4 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 630 bytes | 630.00 KiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Dec32th/laidd-2026.git
   42f7214..a31b4e3  main -> main


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMMPA

sample_smiles = None
for s in data['smiles_train']:
    m = Chem.MolFromSmiles(s)
    if m and 15 <= m.GetNumAtoms() <= 30:
        sample_smiles = s
        break

print("선택된 분자:", sample_smiles)
mol2 = Chem.MolFromSmiles(sample_smiles)

fragments3 = rdMMPA.FragmentMol(mol2, maxCuts=1, resultsAsMols=False)
print("fragment 개수:", len(fragments3))
for core, chain in fragments3[:5]:
    print(f"core: {core}, chain: {chain}")

선택된 분자: O=C1CC[C@@H](C(=O)N[C@H]2C[C@@H]2c2ccccc2)N1
fragment 개수: 3
core: , chain: O=C(N[C@H]1C[C@@H]1c1ccccc1)[*:1].O=C1CC[C@@H]([*:1])N1
core: , chain: O=C1CC[C@@H](C(=O)N[*:1])N1.c1ccc([C@H]2C[C@@H]2[*:1])cc1
core: , chain: O=C1CC[C@@H](C(=O)N[C@H]2C[C@@H]2[*:1])N1.c1ccc([*:1])cc1


In [ ]:
fragments4 = rdMMPA.FragmentMol(mol2, maxCuts=2, resultsAsMols=False)
print("fragment 개수:", len(fragments4))
for core, chain in fragments4[:10]:
    print(f"core: {core}, chain: {chain}")

fragment 개수: 6
core: , chain: O=C(N[C@H]1C[C@@H]1c1ccccc1)[*:1].O=C1CC[C@@H]([*:1])N1
core: O=C(N[*:2])[*:1], chain: O=C1CC[C@@H]([*:1])N1.c1ccc([C@H]2C[C@@H]2[*:2])cc1
core: O=C(N[C@H]1C[C@@H]1[*:2])[*:1], chain: O=C1CC[C@@H]([*:1])N1.c1ccc([*:2])cc1
core: , chain: O=C1CC[C@@H](C(=O)N[*:1])N1.c1ccc([C@H]2C[C@@H]2[*:1])cc1
core: C1[C@H]([*:1])[C@H]1[*:2], chain: O=C1CC[C@@H](C(=O)N[*:1])N1.c1ccc([*:2])cc1
core: , chain: O=C1CC[C@@H](C(=O)N[C@H]2C[C@@H]2[*:1])N1.c1ccc([*:1])cc1


In [ ]:
fragments_with_core = [(core, chain) for core, chain in fragments4 if core]
print(f"core가 있는 조합: {len(fragments_with_core)}개")
for core, chain in fragments_with_core:
    print(f"core: {core}, chain: {chain}")

core가 있는 조합: 3개
core: O=C(N[*:2])[*:1], chain: O=C1CC[C@@H]([*:1])N1.c1ccc([C@H]2C[C@@H]2[*:2])cc1
core: O=C(N[C@H]1C[C@@H]1[*:2])[*:1], chain: O=C1CC[C@@H]([*:1])N1.c1ccc([*:2])cc1
core: C1[C@H]([*:1])[C@H]1[*:2], chain: O=C1CC[C@@H](C(=O)N[*:1])N1.c1ccc([*:2])cc1
